In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error

# data
data = {
    'survived': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
                 1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
    'pclass':   [1, 3, 1, 3, 2, 3, 1, 2, 3, 1,
                 2, 3, 1, 2, 3, 1, 2, 3, 1, 2],
    'age':      [22, 35, np.nan, 27, 14, np.nan, 58, 20, 33, 45,
                 29, np.nan, 61, 18, 24, 37, np.nan, 29, 41, 16],
    'fare':     [150, 7.5, 200, 8.0, 21, 7.2, 300, 13, 8.5, 250,
                 18, 7.8, 350, 16, 9.0, 275, 19, 8.2, 400, 17],
    'sex':      ['male','male','female','female','female','male',
                 'female','male','male','female','female','male',
                 'female','male','male','female','female','male','female','male'],
    'embarked': ['S','S','C','S','C','Q','C','S','Q','C',
                 'S','Q','C','S','S','C','Q','S','C','S']
}

df = pd.DataFrame(data)

In [2]:
# define columns
CATEGORIES = df.columns[[1, 4, 5]].values  # pclass, sex, embarked
NUMS = df.columns[[2, 3]].values            # age, fare

# train test split
train, test = train_test_split(df, test_size=0.2, shuffle=True, 
                                stratify=df.sex, random_state=42)

# pipelines
num_pipe = Pipeline([
    ("median", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("onehot", OneHotEncoder(sparse_output=False))
])

full_pipe = ColumnTransformer([
    ("numeric_cols", num_pipe, NUMS),
    ("categorical_cols", cat_pipe, CATEGORIES)
])

In [ ]:
# prepare training data
X_train = train.drop("survived", axis=1)
y_train = train["survived"]
X_train_prepared = full_pipe.fit_transform(X_train)

# fit model
lin_reg = LinearRegression()
lin_reg.fit(X_train_prepared, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [4]:
# prepare test data and predict
X_test = test.drop("survived", axis=1)
y_test = test["survived"]
X_test_prepared = full_pipe.transform(X_test)
predictions = lin_reg.predict(X_test_prepared)

# threshold predictions for binary output
predictions_binary = (predictions >= 0.5).astype(int)

In [5]:
# evaluate
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"RMSE: {rmse}")
print(f"Predictions: {predictions}")
print(f"Binary predictions: {predictions_binary}")
print(f"Actual: {y_test.values}")

RMSE: 0.8899382217520445
Predictions: [-0.15204217 -0.1353664   0.60293236  0.43379109]
Binary predictions: [0 0 1 0]
Actual: [1 1 0 0]


In [50]:
log_reg = LogisticRegression(l1_ratio=0, C=.001, 
                             solver='liblinear', 
                             max_iter=100, 
                             fit_intercept=True,
                             class_weight='balanced',
                             tol=1e-4
                             )

#In logistic regression. The intercept is the probability of target = 1 given all features/variables are 0. 

log_reg.fit(X_train_prepared,y_train)
log_reg.predict_proba(X=X_test_prepared)

array([[0.50000929, 0.49999071],
       [0.50001346, 0.49998654],
       [0.50054399, 0.49945601],
       [0.50017648, 0.49982352]])

In [51]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


log_predictions = log_reg.predict(X_test_prepared)
accuracy_score(y_true=y_test, y_pred=log_predictions)
print(classification_report(y_true=y_test, y_pred=log_predictions))
confusion_matrix(y_true=y_test, y_pred=log_predictions)

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4



c:\Users\kn806mj\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\kn806mj\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\kn806mj\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

array([[2, 0],
       [2, 0]])

In [ ]:
np.concat([y_test.values, log_predictions]).reshape()

array([[1, 1, 0, 0],
       [0, 0, 0, 1]])

In [49]:
#GridSearchCV

from sklearn.model_selection import GridSearchCV
param_grid = {"C": np.power(10.0,np.arange(-3,3))}
grid_search = GridSearchCV(log_reg, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_prepared, y_train)
print(grid_search.best_params_)


{'C': np.float64(0.001)}
